# Microproyecto 2: Identificación de Relaciones Semánticas con PLN y ML

---

## A. Objetivo
Desarrollar una solución basada en técnicas de **procesamiento de lenguaje natural (PLN)** y **machine learning (ML)** que facilite la interpretación y análisis de información textual para la identificación de relaciones semánticas con los **Objetivos de Desarrollo Sostenible (ODS)**.

---

## B. Conjunto de Datos
El conjunto de datos forma parte del proyecto [OSDG Community Dataset (OSDG-CD)](https://osdg.ai/news/New-release-of-OSDG-Community-dataset) en su versión 2023, que contiene un total de **40.067 textos**, de los cuales **3.000** provienen de fuentes relacionadas con las Naciones Unidas. También contiene documentos públicos, resúmenes de artículos y reportes.

La plataforma reúne investigadores, expertos en la materia y defensores de los ODS de todo el mundo para crear una fuente amplia y precisa de información textual sobre los ODS. Los voluntarios de la comunidad utilizan la plataforma para participar en ejercicios de etiquetado en los que validan la relevancia de cada texto para los ODS basándose en sus conocimientos previos.

> **Traducción y Aumentación de Datos:**
> - Los textos utilizados en este proyecto han sido traducidos al español mediante herramientas como [DeepL](https://www.deepl.com/es/translator).
> - Se realizó aumentación de textos a través de la API de [ChatGPT / OpenAI](https://chat.openai.com/g/g-I1XNbsyDK-api-docs).

---

## C. Actividades a Realizar

1. **Preparación de los textos:**
   - Utilizar el esquema de bolsa de palabras (**BoW**) con pesado **TF-IDF**.
   - Construir un **pipeline** que integre todas las transformaciones y preprocesamientos que se consideren adecuados.

2. **Modelado de Tópicos (LSA):**
   - A partir de la matriz TF-IDF construida, aplicar el algoritmo SVD truncado (`TruncatedSVD` de scikit-learn) para obtener un modelo de tópicos mediante **Análisis Semántico Latente (LSA)**.
   - Explorar un número reducido de componentes (por ejemplo, entre 10 y 20).
   - Para al menos **5 componentes**, identificar y mostrar las palabras con mayor peso a modo de *tópicos*.
   - Interpretar cualitativamente si estos tópicos guardan relación con algunos de los 17 ODS trabajados en el proyecto.

3. **Desarrollo del Modelo de Clasificación:**
   - Construir un modelo de clasificación que permita relacionar un texto con su respectivo ODS.
   - Para manejar la complejidad del espacio de entrada, se puede reutilizar la descomposición SVD (LSA) o aplicar otra técnica de reducción de dimensionalidad pertinente.

4. **Evaluación del Modelo:**
   - Evaluar el modelo con un conjunto de prueba (textos no utilizados durante la etapa de entrenamiento/aprendizaje).

---

## D. Consideraciones
El algoritmo de clasificación a utilizar, así como la técnica de reducción de la dimensionalidad, queda a consideración de cada grupo; sin embargo, **es fundamental justificar la elección de cada técnica**.

---

## E. Entregable
- **Archivos:** Notebook en formatos `.ipynb` y `.html` con el método desarrollado.
- **Documentación:** El notebook debe estar completamente documentado con las justificaciones de las decisiones tomadas en cada paso.
- **Ejecución visible:** Deben ser visibles las salidas y ejecuciones de cada celda.
- **Evidencia práctica:** Para evidenciar el desempeño del método construido, el notebook debe mostrar las clasificaciones para al menos **4 textos del conjunto de test**.
- **Plazo:** Entrega al final de la **Semana 7** en el espacio correspondiente.

---

## F. Criterios de Evaluación

| Actividad | Porcentaje |
| :--- | :---: |
| **Preparación de los datos**, incluyendo la reducción de la dimensionalidad y justificación de decisiones tomadas. | **30%** |
| **Construcción del pipeline** de preparación de datos. | **15%** |
| **Construcción del modelo de clasificación** con el algoritmo seleccionado, búsqueda de hiperparámetros y validación con medidas de evaluación adecuadas (justificando algoritmo, métricas y reducción de dimensionalidad). | **30%** |
| **Evidencia del desempeño** del modelo mostrando clasificaciones sobre un conjunto de textos no utilizados durante el aprendizaje. | **10%** |
| **Construcción del modelo LSA** sobre la matriz TF-IDF e interpretación cualitativa de al menos 5 tópicos frente a los ODS. | **15%** |
| **Total** | **100%** |

---

## G. Bonificación Adicional (Opcional — +15 Puntos)
Con el propósito de fortalecer las competencias en despliegue y aplicación práctica de modelos de ML, se otorgará una bonificación adicional de **15 puntos** a los grupos que implementen el modelo en una aplicación interactiva utilizando **Streamlit**.

### Requisitos de la Aplicación:
- Permitir al usuario ingresar un texto libre.
- Procesar el texto utilizando el mismo pipeline construido en el proyecto.
- Generar como salida la predicción del Objetivo de Desarrollo Sostenible (ODS) correspondiente.
- Ser totalmente funcional y ejecutarse correctamente.

> *Nota:* La bonificación es voluntaria y no reemplaza los criterios de la rúbrica; premia el paso del entorno experimental al despliegue práctico.


## Importe Librerias

In [27]:
from nltk import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import LinearSVC
from stop_words import get_stop_words

import joblib
import pandas as pd
import re as regexExpression
import string
import warnings

warnings.filterwarnings('ignore')

## Carga de datos

In [2]:
raw_data = pd.read_excel('data/Datos_textosODS.xlsx')
raw_data.sample(5)

,textos,ODS
7653,Aunque las declaraciones del Consejo de Seguri...,16
8436,También se comparan los países de la OCDE en r...,3
331,"De hecho, como la UIT ha estado destacando dur...",9
7003,"Sin embargo, estos indicadores tienen limitaci...",6
5766,El tercer nivel consta de varios sistemas de m...,3


Se validan datos nulos y/o duplicados:

In [3]:
print(f'Datos duplicados:\n{raw_data.duplicated().sum()}')
print(f'Datos nulos:\n{raw_data.isna().sum()}')

Datos duplicados:
0
Datos nulos:
textos    0
ODS       0
dtype: int64


No hay neceesidad de eliminar datos nulos o duplicados.

## Funcion para preparar textos

In [ ]:
def prepare_text(text):
    tokenizer = RegexpTokenizer(r'\w+')
    stemmer = PorterStemmer()
    tokens = tokenizer.tokenize(text)
    tokens = [word for word in tokens if word not in stopwords.words('spanish')]
    tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)

# Prueba de la preparacion de textos
random_text = text.sample(1).item()
prepared_text = prepare_text(random_text)
print(f'Texto original:\n{random_text}\nTexto preparado:\n{prepared_text}')

## Separacion del conjunto de datos

In [ ]:
data = raw_data.copy()
data['texto_limpio'] = data['textos'].apply(prepare_text)

text = data['texto_limpio']
labels = data['ODS']

X_train, X_test, y_train, y_test = train_test_split(text, labels, test_size=0.2, random_state=55)

## Preparacion de vectorizacion y reduccion de la dimensionalidad

In [8]:
vectorizer = TfidfVectorizer(preprocessor=prepare_text)
tsvd = TruncatedSVD(n_components=100, random_state=32)

## Busqueda del mejor mocelo con cross validation

Vamos a evalkuar los siguientes modelos

 - Regresion Logistiva
 - Maquina de Soporte Vectorial
 - Gaussian NB

In [ ]:
def find_best_model(vectorizer, tsvd, X_train, y_train):
    # Definimos la estrategia de validación cruzada manteniendo proporción de clases
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=32)
    # 2. Definimos los modelos e hiper parametros a probar
    experimentos = {
        "Linear SVC": {
            "pipeline": Pipeline([
                ("vectorizer", vectorizer),
                ("dimred", tsvd),
                ("model", LinearSVC(max_iter=2000, random_state=32))
            ]),
            "params": {
                "model__C": [0.1, 0.5],
                "dimred__n_components": [100, 150]
            }
        },
        "Logistic Regression": {
            "pipeline": Pipeline([
                ("vectorizer", vectorizer),
                ("dimred", tsvd),
                ("model", LogisticRegression(max_iter=500, random_state=32))
            ]),
            "params": {
                "model__C": [0.5, 1.0],
                "dimred__n_components": [100]
            }
        },
        "Gaussian NB": {
            "pipeline": Pipeline([
                ("vectorizer", vectorizer),
                ("dimred", tsvd),
                ("model", GaussianNB())
            ]),
            "params": {
                "model__var_smoothing": [1e-9]
            }
        }
    }
    # Ejecutamos GridSearch para cada pipeline
    resultados = []
    mejores_modelos = {}
    for nombre, config in experimentos.items():
        print(f"Optimizando {nombre}...")
        
        # Se usa f1_weighted dado el desbalance entre los diferentes ODS
        grid = GridSearchCV(
            estimator=config["pipeline"],
            param_grid=config["params"],
            cv=cv,
            scoring="f1_weighted",
            n_jobs=1,
            verbose=1
        )
        
        grid.fit(X_train, y_train)
        mejores_modelos[nombre] = grid.best_estimator_
        
        # Evaluar en el conjunto de prueba no visto
        test_score = grid.score(X_test, y_test)
        
        resultados.append({
            "Modelo": nombre,
            "Mejor F1 (CV Train)": round(grid.best_score_, 4),
            "F1 Test (X_test)": round(test_score, 4),
            "Mejores Hiperparámetros": grid.best_params_
        })
    # 4. Tabla comparativa final
    df_resultados = pd.DataFrame(resultados).sort_values(by="F1 Test (X_test)", ascending=False)
    return df_resultados

cv_report = find_best_model(vectorizer, tsvd, X_train, y_train)
cv_report

Optimizando Linear SVC...
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Optimizando Logistic Regression...
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Optimizando Gaussian NB...
Fitting 5 folds for each of 1 candidates, totalling 5 fits


,Modelo,Mejor F1 (CV Train),F1 Test (X_test),Mejores Hiperparámetros
0,Linear SVC,0.8622,0.8569,"{'dimred__n_components': 150, 'model__C': 0.5}"
1,Logistic Regression,0.8402,0.8378,"{'dimred__n_components': 100, 'model__C': 1.0}"
2,Gaussian NB,0.7819,0.7832,{'model__var_smoothing': 1e-09}


## Construccion del pipeline con el mejor modelo encontrado

In [ ]:
steps = [
    ("vectorizer", vectorizer),
    ("dimred", tsvd),
    ("model", LinearSVC(C=0.5, random_state=32, max_iter=2000)),
]
pipeline = Pipeline(steps)

## Validacion de rendimiento del mejor modelo

In [24]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
matrix = pd.DataFrame(confusion_matrix(y_test, y_pred, labels=labels.unique()))
reporte = pd.DataFrame(classification_report(y_test, y_pred, output_dict=True))

In [25]:
matrix

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,192,0,0,0,0,0,0,0,0,1,0,0,1,0,2,0
1,0,131,4,0,1,1,1,1,0,0,2,0,0,2,0,1
2,0,0,69,0,0,1,3,2,1,1,6,1,0,0,1,0
3,3,0,0,214,3,0,0,0,0,0,0,0,0,0,0,0
4,5,0,0,6,203,0,2,0,3,0,0,0,0,0,0,0
5,1,0,3,2,0,58,5,1,0,2,2,0,3,2,0,0
6,2,2,0,3,4,0,162,2,3,2,0,0,0,0,0,0
7,1,5,3,5,1,0,5,87,2,1,3,0,0,0,0,3
8,6,0,0,3,6,0,3,6,41,4,8,0,9,0,2,0
9,0,1,1,3,0,2,1,1,2,83,0,0,0,0,3,0


In [26]:
reporte.transpose()

,precision,recall,f1-score,support
1,0.805825,0.855670,0.830000,97.000000
2,0.840580,0.734177,0.783784,79.000000
3,0.848168,0.900000,0.873315,180.000000
4,0.884793,0.979592,0.929782,196.000000
5,0.926941,0.926941,0.926941,219.000000
6,0.867550,0.909722,0.888136,144.000000
7,0.811765,0.920000,0.862500,150.000000
8,0.650794,0.465909,0.543046,88.000000
9,0.704918,0.605634,0.651515,71.000000
10,0.843137,0.614286,0.710744,70.000000


# Exportar modelo

In [28]:
joblib.dump(pipeline, 'modelo_ods.joblib', compress=3)

['modelo_ods.joblib']